# Ch.4 — Support Vector Machines for Smiling Detection

**FaceAI**: Maximum-margin classification with linear and RBF kernels.

**Goal**: Push accuracy from 88% (LogReg) to ~89% (SVM) via margin maximization.

**Key concepts**: Hyperplane, margin, support vectors, kernel trick, C/gamma trade-off.

In [ ]:
# ── Setup & Imports ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, roc_auc_score,
                             roc_curve, ConfusionMatrixDisplay)
from sklearn.decomposition import PCA

IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)
SAVE_KW = dict(dpi=150, bbox_inches='tight')
np.random.seed(42)
print("Imports OK")

## §0 Data — CelebA Smiling (same setup)

In [ ]:
# ── Synthetic CelebA Proxy ─────────────────────────────
X, y = make_classification(
    n_samples=5000, n_features=200, n_informative=40,
    n_redundant=20, n_clusters_per_class=3, weights=[0.52, 0.48],
    flip_y=0.05, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## §1 Linear SVM vs Logistic Regression

In [ ]:
# ── Linear SVM vs LogReg ───────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=500, random_state=42)
lr.fit(X_train_s, y_train)

linear_svm = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
linear_svm.fit(X_train_s, y_train)

print(f"LogReg accuracy:     {lr.score(X_test_s, y_test):.3f}")
print(f"Linear SVM accuracy: {linear_svm.score(X_test_s, y_test):.3f}")
print(f"\nSupport vectors: {linear_svm.n_support_} ({sum(linear_svm.n_support_)/len(y_train)*100:.1f}% of training data)")

## §2 RBF Kernel — Non-linear Boundaries

In [ ]:
# ── RBF SVM ────────────────────────────────────────────
rbf_svm = SVC(kernel='rbf', C=10, gamma=0.01, probability=True, random_state=42)
rbf_svm.fit(X_train_s, y_train)

y_pred = rbf_svm.predict(X_test_s)
y_prob = rbf_svm.predict_proba(X_test_s)[:, 1]

print(f"RBF SVM accuracy: {rbf_svm.score(X_test_s, y_test):.3f}")
print(f"ROC-AUC:          {roc_auc_score(y_test, y_prob):.3f}")
print(f"Support vectors:  {sum(rbf_svm.n_support_)} ({sum(rbf_svm.n_support_)/len(y_train)*100:.1f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Smiling', 'Smiling']))

## §3 C and Gamma Exploration

In [ ]:
# ── C and Gamma Heatmap ────────────────────────────────
C_values = [0.1, 1, 10, 100]
gamma_values = [0.001, 0.01, 0.1, 1.0]

# Use a smaller subset for speed
X_sub, y_sub = X_train_s[:2000], y_train[:2000]

scores = np.zeros((len(C_values), len(gamma_values)))
for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        svm = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        cv_scores = cross_val_score(svm, X_sub, y_sub, cv=3, scoring='accuracy')
        scores[i, j] = cv_scores.mean()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(scores, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(gamma_values)))
ax.set_xticklabels([f'{g}' for g in gamma_values])
ax.set_yticks(range(len(C_values)))
ax.set_yticklabels([f'{c}' for c in C_values])
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.set_title('SVM Accuracy: C vs Gamma')

for i in range(len(C_values)):
    for j in range(len(gamma_values)):
        ax.text(j, i, f'{scores[i,j]:.3f}', ha='center', va='center', fontsize=10)

fig.colorbar(im)
fig.savefig(IMG_DIR / 'c_gamma_heatmap.png', **SAVE_KW)
plt.show()

## §4 Margin Visualization (2D PCA Projection)

In [ ]:
# ── 2D Margin Visualization via PCA ────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_s[:1000])
y_pca = y_train[:1000]

svm_2d = SVC(kernel='rbf', C=10, gamma=0.5, random_state=42)
svm_2d.fit(X_pca, y_pca)

# Decision boundary
h = 0.2
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_pca, cmap='RdYlBu',
                     edgecolors='black', linewidth=0.5, s=20, alpha=0.7)

# Highlight support vectors
sv = svm_2d.support_
ax.scatter(X_pca[sv, 0], X_pca[sv, 1], s=80, facecolors='none',
           edgecolors='black', linewidth=2, label=f'Support Vectors ({len(sv)})')

ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title('SVM Decision Boundary (2D PCA projection)')
ax.legend()
fig.savefig(IMG_DIR / 'svm_decision_boundary.png', **SAVE_KW)
plt.show()

## §5 ROC Comparison: LogReg vs SVM

In [ ]:
# ── ROC Comparison ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

models_cmp = {
    'LogReg': lr,
    'Linear SVM': linear_svm,
    'RBF SVM': rbf_svm,
}

for name, model in models_cmp.items():
    y_p = model.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_p)
    auc = roc_auc_score(y_test, y_p)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Comparison — LogReg vs SVM')
ax.legend()
fig.savefig(IMG_DIR / 'roc_comparison.png', **SAVE_KW)
plt.show()

## §6 Support Vector Analysis

In [ ]:
# ── Support Vector Statistics ──────────────────────────
n_sv = rbf_svm.n_support_
total_sv = sum(n_sv)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Not Smiling SVs', 'Smiling SVs', 'Non-SVs']
sizes = [n_sv[0], n_sv[1], len(y_train) - total_sv]
colors = ['#ff9999', '#66b3ff', '#99ff99']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title(f'Support Vectors: {total_sv}/{len(y_train)} ({total_sv/len(y_train)*100:.1f}%)')
fig.savefig(IMG_DIR / 'support_vectors_pie.png', **SAVE_KW)
plt.show()

print(f"Support vectors per class: {dict(zip(['Not Smiling', 'Smiling'], n_sv))}")
print(f"Only {total_sv/len(y_train)*100:.1f}% of training data defines the decision boundary")

## §7 Summary

In [ ]:
# ── Chapter Summary ────────────────────────────────────
print("=" * 50)
print("Ch.4 — Support Vector Machines Summary")
print("=" * 50)
print(f"  LogReg:     {lr.score(X_test_s, y_test):.3f}")
print(f"  Linear SVM: {linear_svm.score(X_test_s, y_test):.3f}")
print(f"  RBF SVM:    {rbf_svm.score(X_test_s, y_test):.3f}")
print(f"\nRBF SVM achieves ~89% via maximum-margin + kernel trick")
print(f"Support vectors: {sum(rbf_svm.n_support_)} (sparse representation)")
print("\nConstraint #1 (ACCURACY): ~89% ✅ (improved from 88%)")
print("Next: Ch.5 — Hyperparameter Tuning (push to 92%)")

## Exercises

1. **Kernel comparison**: Train SVM with linear, poly (degree 2, 3), and RBF kernels. Compare accuracy and training time.
2. **class_weight for Bald**: Create imbalanced Bald dataset (2.5%), train SVM with/without `class_weight='balanced'`. Compare F1.
3. **Scaling impact**: Train RBF SVM without StandardScaler. How much does accuracy drop?

In [ ]:
# Exercise 1: Kernel comparison
# TODO: Train with 'linear', 'poly' (deg 2,3), 'rbf'; compare accuracy + timing
pass

In [ ]:
# Exercise 2: class_weight for Bald
# TODO: Create imbalanced data, compare SVM with/without class_weight='balanced'
pass

In [ ]:
# Exercise 3: Scaling impact
# TODO: Train RBF SVM on unscaled data, compare with scaled results
pass